# Model Comparison — Heart Disease Classifier
Loads all MLflow runs from the `classification-comparison` experiment and compares models visually.

In [ ]:
import os
import tempfile
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from mlflow.tracking import MlflowClient

# Works whether the notebook is run from the project root or notebooks/ subdirectory
db_path = "mlflow.db"
if not os.path.exists(db_path):
    db_path = "../mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{os.path.abspath(db_path)}")
client = MlflowClient()
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
experiment = client.get_experiment_by_name("classification-comparison")
if experiment is None:
    raise RuntimeError("Experiment 'classification-comparison' not found. Run main.py first.")

raw = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

metric_cols = ["metrics.accuracy", "metrics.precision", "metrics.recall", "metrics.f1", "metrics.roc_auc"]
runs_df = (
    raw[["run_id", "tags.mlflow.runName"] + metric_cols]
    .rename(columns={"tags.mlflow.runName": "model", **{c: c.split(".")[1] for c in metric_cols}})
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)
runs_df

## Metrics Comparison Table

In [ ]:
(
    runs_df
    .set_index("model")
    .drop(columns="run_id")
    .style
    .format("{:.4f}")
    .background_gradient(cmap="YlGn")
    .set_caption("All metrics — higher is better")
)

## Grouped Bar Chart — Accuracy, F1, ROC-AUC

In [ ]:
plot_df = runs_df.set_index("model")[["accuracy", "f1", "roc_auc"]]

ax = plot_df.plot(kind="bar", figsize=(10, 5), rot=0, width=0.6)
ax.set_title("Model Comparison: Accuracy, F1, ROC-AUC", fontsize=13)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.legend(loc="lower right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
plt.tight_layout()
plt.show()

## Confusion Matrices (side by side)

In [ ]:
n = len(runs_df)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, runs_df.iterrows()):
    with tempfile.TemporaryDirectory() as tmpdir:
        local_path = client.download_artifacts(row["run_id"], "confusion_matrix.png", tmpdir)
        img = mpimg.imread(local_path)
    ax.imshow(img)
    ax.set_title(row["model"], fontsize=12)
    ax.axis("off")

plt.suptitle("Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()